# Money flow: COT positioning

## What
CFTC Commitment of Traders (legacy futures) for wheat. Net non-commercial positioning and a 26-week z-score.

## Why this model
Net positioning (`noncomm long − noncomm short`) is a crowding measure. A 26-week z-score asks whether today's net is unusual versus the last half-year of the same contract — not a forecast.

## How to rerun
Needs only `FINUTIES_API_KEY` in `notebooks/.env`. Run top to bottom.

**Endpoint (verified 200):** `GET /api/v1/cftc/legacy_futures-facts?commodity=WHEAT`

**Columns used:** `commodity_name`, `contract_market_name`, `report_date_as_yyyy_mm_dd`, `noncomm_positions_long_all`, `noncomm_positions_short_all`

Unfiltered `limit=500` returns only the latest report week. Filter by `commodity` (or `cftc_contract_market_code`) to get history.

In [1]:
from pathlib import Path
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from dotenv import load_dotenv


def resolve_notebooks_env(start_dir: Path) -> Path:
    current = start_dir.resolve()
    for candidate_root in [current, *current.parents]:
        if candidate_root.name == "notebooks":
            env_path = candidate_root / ".env"
            if env_path.exists():
                return env_path
        nested_env = candidate_root / "notebooks" / ".env"
        if nested_env.exists():
            return nested_env
    raise FileNotFoundError(
        "Missing notebooks/.env. Copy notebooks/.env.example and set FINUTIES_API_KEY. "
        "A sandbox key is POST https://data.finuties.com/api/v1/auth/sandbox"
    )


def require_frame(df: pd.DataFrame, required: list[str], min_rows: int = 1) -> None:
    if df.empty:
        raise AssertionError("Expected a non-empty frame from the FinUties API.")
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise AssertionError(f"Missing required columns {missing}. Got {list(df.columns)}")
    if len(df) < min_rows:
        raise AssertionError(f"Expected at least {min_rows} rows, got {len(df)}")


def require_finite(series: pd.Series, name: str) -> None:
    numeric = pd.to_numeric(series, errors="coerce")
    valid = numeric.dropna()
    if valid.empty:
        raise AssertionError(f"{name} has no numeric values")
    if not np.isfinite(valid.to_numpy()).all():
        raise AssertionError(f"{name} contains non-finite values")


load_dotenv(resolve_notebooks_env(Path.cwd()))
API_ORIGIN = os.getenv("FINUTIES_API_ORIGIN", "https://data.finuties.com").rstrip("/")
API_KEY = os.getenv("FINUTIES_API_KEY", "").strip()
if not API_KEY:
    raise ValueError(
        "Missing FINUTIES_API_KEY. Copy notebooks/.env.example to notebooks/.env "
        "and set a key from POST /api/v1/auth/sandbox"
    )

HEADERS = {"Authorization": f"Bearer {API_KEY}"}
TIMEOUT_SECONDS = 45


def finuties_get(endpoint: str, params: dict | None = None):
    response = requests.get(
        f"{API_ORIGIN}{endpoint}",
        headers=HEADERS,
        params=params or {},
        timeout=TIMEOUT_SECONDS,
    )
    response.raise_for_status()
    return response.json()


def normalize_rows(payload) -> list[dict]:
    if isinstance(payload, list):
        return [row for row in payload if isinstance(row, dict)]
    if isinstance(payload, dict):
        for key in ("items", "data", "rows", "results"):
            rows = payload.get(key)
            if isinstance(rows, list):
                return [row for row in rows if isinstance(row, dict)]
    return []


In [2]:
COT_ENDPOINT = "/api/v1/cftc/legacy_futures-facts"
payload = finuties_get(COT_ENDPOINT, {"commodity": "WHEAT", "limit": 300})
df = pd.DataFrame(normalize_rows(payload))
require_frame(
    df,
    [
        "commodity_name",
        "contract_market_name",
        "report_date_as_yyyy_mm_dd",
        "noncomm_positions_long_all",
        "noncomm_positions_short_all",
    ],
    min_rows=26,
)

df["report_date"] = pd.to_datetime(df["report_date_as_yyyy_mm_dd"], errors="coerce")
df["long"] = pd.to_numeric(df["noncomm_positions_long_all"], errors="coerce")
df["short"] = pd.to_numeric(df["noncomm_positions_short_all"], errors="coerce")
df = df.dropna(subset=["report_date", "long", "short", "contract_market_name"]).copy()
df["net_positioning"] = df["long"] - df["short"]
require_finite(df["net_positioning"], "net_positioning")

print(f"Rows: {len(df):,}  contracts: {df['contract_market_name'].nunique()}")
print(f"Date range: {df['report_date'].min().date()} → {df['report_date'].max().date()}")
df[["report_date", "contract_market_name", "long", "short", "net_positioning"]].head()

Rows: 300  contracts: 3
Date range: 2024-09-03 → 2026-08-11


,report_date,contract_market_name,long,short,net_positioning
0,2026-08-11,WHEAT-HRW,71932,58718,13214
1,2026-08-11,WHEAT-HRSpring,19038,9379,9659
2,2026-08-11,WHEAT-SRW,114979,139889,-24910
3,2026-08-04,WHEAT-HRSpring,19489,10762,8727
4,2026-08-04,WHEAT-HRW,73792,56135,17657


In [3]:
window = 26
contract = df["contract_market_name"].value_counts().idxmax()
series = (
    df[df["contract_market_name"] == contract]
    .drop_duplicates(subset=["report_date"])
    .sort_values("report_date")
    .reset_index(drop=True)
    .copy()
)
require_frame(series, ["report_date", "net_positioning"], min_rows=window)

roll_mean = series["net_positioning"].rolling(window=window, min_periods=window).mean()
roll_std = series["net_positioning"].rolling(window=window, min_periods=window).std()
series["zscore_26w"] = (series["net_positioning"] - roll_mean) / roll_std.replace(0, np.nan)
scored = series.dropna(subset=["zscore_26w"])
require_frame(scored, ["zscore_26w"], min_rows=1)
require_finite(scored["zscore_26w"], "zscore_26w")
assert np.isfinite(float(scored["zscore_26w"].iloc[-1]))

series["signal_band"] = pd.cut(
    series["zscore_26w"],
    bins=[-np.inf, -2, 2, np.inf],
    labels=["short_extreme", "neutral", "long_extreme"],
)
print(f"Contract: {contract}  scored weeks: {len(scored)}")
series[["report_date", "contract_market_name", "net_positioning", "zscore_26w", "signal_band"]].tail(12)

Contract: WHEAT-HRW  scored weeks: 75


,report_date,contract_market_name,net_positioning,zscore_26w,signal_band
88,2026-05-19,WHEAT-HRW,5416,0.582396,neutral
89,2026-05-26,WHEAT-HRW,-1734,-0.230036,neutral
90,2026-06-02,WHEAT-HRW,-9002,-1.059562,neutral
91,2026-06-09,WHEAT-HRW,-21263,-2.184041,short_extreme
92,2026-06-16,WHEAT-HRW,-9238,-0.906175,neutral
93,2026-06-23,WHEAT-HRW,-15127,-1.459977,neutral
94,2026-06-30,WHEAT-HRW,-10339,-0.970312,neutral
95,2026-07-07,WHEAT-HRW,-6013,-0.521666,neutral
96,2026-07-14,WHEAT-HRW,1769,0.246666,neutral
97,2026-07-28,WHEAT-HRW,15958,1.509472,neutral


In [4]:
fig, axes = plt.subplots(2, 1, figsize=(16, 6), sharex=True)

axes[0].plot(
    series["report_date"],
    series["net_positioning"],
    color="#1f77b4",
    linewidth=1.8,
    label="Net non-commercial positioning",
)
axes[0].set_title(f"COT net positioning — {contract}")
axes[0].set_ylabel("Contracts")
axes[0].legend(loc="upper left")

axes[1].plot(
    series["report_date"],
    series["zscore_26w"],
    color="#d62728",
    linewidth=1.8,
    label="26-week z-score",
)
axes[1].axhline(2, linestyle="--", linewidth=1, color="#444444", label="+2 / −2 band")
axes[1].axhline(-2, linestyle="--", linewidth=1, color="#444444")
axes[1].set_ylabel("Z-score (unitless)")
axes[1].set_xlabel("Report date")
axes[1].legend(loc="upper left")

fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## Caveats

- One contract only. Other wheat contracts (HRW, HRSpring, SRW) can disagree.
- A large |z-score| is a positioning extreme, not a trade signal.
- CFTC revisions and contract-spec changes move history.
- This is not investment advice.